In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import (
    DEFAULT_CUBE_SIZES,
    BereaPatchDataset,
    DigitalCorePipeline,
    TopologyAdaptiveRoutedUNet3D,
    PoreNetworkPermeabilityModel,
    TOPOLOGY_FEATURE_DIM,
    check_required_dependencies,
)

check_required_dependencies()
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


In [3]:
SEG_CKPT_CANDIDATES = [
    ROOT / "models" / "segmentation_best.pth",
    ROOT / "models" / "topology_quick_64.pth",
]
GNN_CKPT = ROOT / "models" / "gnn_pnm_best.pth"

def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def looks_like_segmentation_checkpoint(checkpoint):
    state = checkpoint.get("model_state_dict", checkpoint)
    return any(key.startswith("enc1.") or key.startswith("out_conv.") for key in state)

seg_checkpoint = None
SEG_CKPT = None
for candidate in SEG_CKPT_CANDIDATES:
    if not candidate.exists():
        continue
    checkpoint = load_checkpoint(candidate)
    if looks_like_segmentation_checkpoint(checkpoint):
        SEG_CKPT = candidate
        seg_checkpoint = checkpoint
        break

if seg_checkpoint is None:
    raise FileNotFoundError("No valid segmentation checkpoint found in models/. Re-run 01_train_segmentation.ipynb.")

ph_dim = int(seg_checkpoint.get("ph_dim", TOPOLOGY_FEATURE_DIM))
seg_model = TopologyAdaptiveRoutedUNet3D(
    base_channels=seg_checkpoint.get("base_channels", 16),
    ctx_dim=seg_checkpoint.get("ctx_dim", 64),
    ph_dim=ph_dim,
    topology_dim=ph_dim,
).to(device)
seg_model.load_state_dict(seg_checkpoint["model_state_dict"])

gnn_checkpoint = load_checkpoint(GNN_CKPT)
if gnn_checkpoint.get("checkpoint_type") not in (None, "gnn_pnm"):
    raise ValueError(f"Unexpected GNN checkpoint type: {gnn_checkpoint.get('checkpoint_type')}")
graph_model = PoreNetworkPermeabilityModel(
    node_in=gnn_checkpoint["node_in"],
    edge_in=gnn_checkpoint["edge_in"],
    hidden=gnn_checkpoint.get("hidden", 64),
    layers=gnn_checkpoint.get("layers", 3),
    mu=1.0e-3,
).to(device)
graph_model.load_state_dict(gnn_checkpoint["model_state_dict"])
print("loaded segmentation checkpoint:", SEG_CKPT)
print("loaded GNN checkpoint:", GNN_CKPT)


In [ ]:
CUBE_SIZE = 128  # change to 64 or 192 to inspect another prepared scale
VOXEL_SIZE_M = 2.25e-6
dataset = BereaPatchDataset(ROOT, split="val", cube_size=CUBE_SIZE, use_raw_gray=False, noise_types=["none"], balance=False, return_topology=True)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
cube = batch["x"][0].numpy()
coord = batch["coord"][0].tolist()
rock = batch["rock"][0]
cube_size = int(batch["cube_size"][0])
DOMAIN_SIZE = (cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M)
print("rock:", rock, "coord:", coord, "cube:", cube.shape)


In [ ]:
pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    graph_model=graph_model,
    device=device,
    threshold=0.5,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)

result = pipeline.run_cube(
    cube,
    input_is_pore_mask=False,
    domain_size=DOMAIN_SIZE,
    include_ph=True,
    compute_openpnm_baseline=True,
)

k_gnn = result.permeability.k_gnn_pnm.detach().cpu().numpy()
print("network:", result.network.metadata)
print("GNN+PNM k:", k_gnn)
print("OpenPNM k:", result.permeability.k_openpnm)


In [ ]:
row = {
    "rock": rock,
    "cube_size": cube_size,
    "z": coord[0],
    "y": coord[1],
    "x": coord[2],
    "num_pores": result.network.metadata["num_pores"],
    "num_throats": result.network.metadata["num_throats"],
    "gnn_pnm_kx": float(k_gnn[0]),
    "gnn_pnm_ky": float(k_gnn[1]),
    "gnn_pnm_kz": float(k_gnn[2]),
}
if result.permeability.k_openpnm is not None:
    row.update({f"openpnm_{k}": v for k, v in result.permeability.k_openpnm.items()})

df = pd.DataFrame([row])
path = OUT_DIR / "full_pipeline_summary.csv"
df.to_csv(path, index=False)
print("saved:", path)
df
